○ XGBoost와 LightGBM은 둘 다 “그래디언트 부스팅(Gradient Boosting)” 계열의 머신러닝 모델
  - 약한 예측기(주로 의사결정나무 Decision Tree)를 여러 개 만들어서, 예측 오차를 줄여 나가는 방식

## 1. 공통 뿌리: 그래디언트 부스팅 (Gradient Boosting)
  - 기본 아이디어 
    1) 첫 번째 나무(모델)로 예측을 해본다.
    2) 틀린 부분(오차, residual)을 다음 나무가 학습한다.
    3) 이런 식으로 나무를 계속 추가하면서 점점 오차를 줄여 나간다.
    4) "앙상블(Ensemble)" 기법의 하나.
    5) 성격상 단순한 선형 회귀나 로지스틱보다 훨씬 복잡한 패턴을 잡아낼 수 있다.

## 2. XGBoost (Extreme Gradient Boosting)
  - 핵심 특징
    1) “한 그루 한 그루 나무를 순차적으로” 학습한다.
    2) 오차를 줄이는 데 집중하면서, 과적합을 막기 위해 규제(regularization, L1/L2 penalty) 를 강화.
    3) 병렬화(parallelization)가 지원되지만, 본질적으로는 순차적 학습.

  - 분석/예측 원리
    1) 예측값을 낸 뒤, 그 오차에 해당하는 그래디언트(기울기) 를 계산.
    2) 그 기울기를 가장 잘 설명할 수 있는 새로운 나무를 만든다.
    3) 이런 식으로 “기울기 정보”를 반영한 나무들을 점점 쌓는다.

  - 장점
    1) 성능이 우수하고, 예측력이 강하다.
    2) 정규화 기능이 내장 → 과적합 방지.
    3) 표 형식(tabular) 데이터에서 캐글 같은 대회에서 자주 우승.

  - 단점
    1) 대규모 데이터에서는 학습 시간이 길다.
    2) 메모리 사용량이 크다.

## 3. LightGBM (Light Gradient Boosting Machine)

- 핵심 특징
  1) 학습을 레벨별(level-wise) 이 아닌 리프별(leaf-wise)로 한다.
  2) 손실 감소가 큰 리프를 먼저 확장해서 더 세밀하게 학습한다.
  3) Histogram 기반 학습으로 연속형 값을 bin(구간)으로 나눠 계산 효율을 극대화한다.
  4) 데이터가 크고 feature가 많을수록 속도/성능 차이가 크게 난다.

- 분석/예측 원리
  1) 기본 원리는 그래디언트 부스팅과 동일하다.
  2) 분할 방식을 리프 중심으로 하여 복잡한 데이터 패턴을 더 잘 잡는다.
  3) 따라서 예측력은 강력하지만, 작은 데이터에서는 과적합 위험이 더 크다.

- 장점
  1) 매우 빠르다. XGBoost 대비 학습 속도가 몇 배 이상 빠르다.
  2) 메모리 효율적이다. 대규모 데이터셋에 최적이다.
  3) 복잡한 패턴까지 잡아내는 강력한 성능을 보인다.

- 단점
  1) Leaf-wise 확장 특성 때문에 작은 데이터셋에서는 과적합 발생 가능성이 높다.
  2) 튜닝(early stopping, max_depth 조정 등)이 필수다.
  3) 잘못 쓰면 성능이 오히려 떨어질 수 있다.

## Gradient Boosting 모델 비교

| 구분 | XGBoost | LightGBM | CatBoost |
|------|---------|----------|----------|
| **출시 배경** | 2014, 대회 성능 극대화 목표 | 2016, 대규모 데이터 효율 개선 (MS) | 2017, 범주형 데이터 처리 최적화 (Yandex) |
| **학습 방식** | Level-wise (균형적으로 트리 성장) | Leaf-wise (손실 큰 리프부터 성장) | Ordered Boosting (데이터 순서 고려) |
| **데이터 처리** | 원시값 기반 계산 | Histogram binning (속도↑ 메모리↓) | 범주형 데이터 자동 인코딩 (Ordered Target Statistics) |
| **분석/예측 원리** | 오차(gradient)를 줄이는 방향으로 순차적 나무 생성 | 리프 확장 중심 → 복잡한 패턴도 잘 포착 | 범주형 변수 직접 학습, 데이터 누수 방지 |
| **장점** | - 안정적 성능<br>- 정규화 내장(L1/L2) → 과적합 방지<br>- 중소규모 데이터 강력 | - 학습 속도 매우 빠름<br>- 메모리 효율↑<br>- 대규모/고차원 데이터 최적 | - 범주형 feature 자동 처리<br>- 전처리 부담 적음<br>- 과적합 억제(Ordered Boosting) |
| **단점** | - 대규모 데이터 느림<br>- 메모리 사용 큼 | - 작은 데이터 과적합 위험↑<br>- 튜닝 필수 | - 커뮤니티/레퍼런스 적음<br>- LightGBM보다 느릴 수 있음 |
| **적합한 상황** | - 100만 건 이하 데이터<br>- 안정적이고 해석력 중요할 때 | - 100만~수천만 건 이상 대규모 데이터<br>- 실시간 예측/경고 시스템 | - 범주형 변수 많은 데이터<br>- 전처리 최소화 필요할 때 |
| **혈당 예측 적용** | - 환자 수 적을 때 안정적<br>- “안전빵” 모델 | - 수천 명 이상 환자, 대규모 로그 데이터<br>- 실시간 혈당 경고 시스템 | - 건강검진 CSV처럼 범주형 변수(성별, 흡연/음주 여부 등) 많은 경우 강점 |

## 5. 실제 혈당 예측 시 적용 차이

- XGBoost
  1) 안정적이고 예측도 괜찮다.
  2) 특히 환자 수가 적거나 데이터가 크지 않을 때 유리하다.
  3) “안전빵” 느낌의 모델이다.

- LightGBM
  1) 환자 수가 많고 데이터가 방대하다면 최적이다.
  2) “실시간 혈당 경고 시스템”처럼 속도가 중요한 경우 적합하다.
  3) 대신 과적합 방지(early stopping, max_depth 등) 세팅이 필수다.